# An-Ra V4 — Google Colab T4 Preflight and Protected Trainer

This is the Google Colab replacement for the P100 training path. It has no Kaggle imports or Kaggle input/output paths.

The default `RUN_TRAINING = False` mode performs a non-destructive preflight and completes cleanly when Drive, the repository, the checkpoint, or a GPU is unavailable. Set `RUN_TRAINING = True` only after the preflight has found the canonical `ANRA_T4_TRAINING_HOME` folder, the signed identity, and the required data files.

The optional training path uses one canonical writer, verified full-resume checkpoint bytes, an immutable data window, and mounted-Drive checkpoint durability. Do not run two training-enabled copies concurrently.

References used for the Colab and runtime checks:
- https://colab.research.google.com/notebooks/io.ipynb
- https://docs.pytorch.org/docs/stable/cuda.html

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = os.environ.get('ANRA_REPO_REF', 'iterate500')
REPO = Path('/content/anra')
SCRATCH = Path('/content/anra_colab_scratch')
DRIVE_ROOT = Path('/content/drive')
TRAINING_HOME_NAME = 'ANRA_T4_TRAINING_HOME'
WORKER_ID = os.environ.get('ANRA_WORKER_ID', 'colab-t4-preflight')
RUN_TRAINING = False
INSTALL_PACKAGE = False
VERIFY_CHECKPOINT_HASH = False
TRAINING_MINUTES = 150
DRAIN_RESERVE_MINUTES = 30
SESSION_BUDGET_MINUTES = int(TRAINING_MINUTES) + int(DRAIN_RESERVE_MINUTES)
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
PACK_CATALOG = [
    {
        'name': 'v4_phase_a_170m_seed1301',
        'start_token': 0,
        'end_token': 170000000,
        'archive_sha256': '07f01bf4809667acc670eb9c94dfab38d28522d7bfc2d4c930e71898cff86ee7',
        'files': [
            {'name': 'v4_phase_a_170m_seed1301.tar.gz.part00', 'size': 83886080, 'sha256': '9efe814598f52275dee15cb70e981e1bb375e24dbf97f39788aa4c84498f33f0'},
            {'name': 'v4_phase_a_170m_seed1301.tar.gz.part01', 'size': 63233323, 'sha256': 'c073e325d2fbe09fe4afefe75c251db59540ea3358e34d8a184d7bb2831e0f6a'},
        ],
    },
    {
        'name': 'v4_phase_a_cont_170m_to_500m_seed1301',
        'start_token': 170000000,
        'end_token': 500000000,
        'archive_sha256': '330ac75a2dbe20a3bd6608adf1c7325fd4a8bf7e81be546ccd8e5f93877f4888',
        'files': [
            {'name': 'v4_phase_a_cont_170m_to_500m_seed1301.tar.gz', 'size': 276647985, 'sha256': '330ac75a2dbe20a3bd6608adf1c7325fd4a8bf7e81be546ccd8e5f93877f4888'},
        ],
    },
]
STATUS = {'drive_mounted': False, 'repo_ready': False, 'package_ready': False, 'gpu_ready': False, 'training_home': None, 'checkpoint_ready': False, 'signing_ready': False, 'training_started': False}
print({'worker': WORKER_ID, 'repo_ref': REPO_REF, 'run_training': RUN_TRAINING, 'session_minutes': SESSION_BUDGET_MINUTES})

In [ ]:
GPU_NAME = 'unavailable'
GPU_MEMORY_GIB = 0.0
CUDA_READY = False
TORCH_ERROR = ''
try:
    import torch
    CUDA_READY = bool(torch.cuda.is_available())
    if CUDA_READY:
        GPU_NAME = str(torch.cuda.get_device_name(0))
        GPU_MEMORY_GIB = float(torch.cuda.get_device_properties(0).total_memory) / (1024 ** 3)
except Exception as exc:
    TORCH_ERROR = f'{type(exc).__name__}: {exc}'
STATUS['gpu_ready'] = CUDA_READY
DRIVE_MOUNTED = False
DRIVE_ERROR = ''
try:
    from google.colab import drive as colab_drive
    colab_drive.mount(str(DRIVE_ROOT), force_remount=False)
    DRIVE_MOUNTED = DRIVE_ROOT.is_dir()
except Exception as exc:
    DRIVE_ERROR = f'{type(exc).__name__}: {exc}'
STATUS['drive_mounted'] = DRIVE_MOUNTED
print({'cuda_ready': CUDA_READY, 'gpu': GPU_NAME, 'memory_gib': round(GPU_MEMORY_GIB, 2), 'drive_mounted': DRIVE_MOUNTED, 'torch_error': TORCH_ERROR, 'drive_error': DRIVE_ERROR})

In [ ]:
REPO_READY = False
PACKAGE_READY = False
COMMIT = ''
REPO_ERROR = ''
if (REPO / '.git').exists():
    REPO_READY = True
elif REPO.exists():
    REPO_ERROR = f'refusing to remove existing non-checkout path: {REPO}'
else:
    clone_command = ['git', '-c', 'http.version=HTTP/1.1', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_REF, REPO_URL, str(REPO)]
    for attempt in range(1, 4):
        result = subprocess.run(clone_command, text=True, capture_output=True)
        if result.returncode == 0:
            REPO_READY = True
            break
        REPO_ERROR = result.stderr.strip() or result.stdout.strip() or f'git exit {result.returncode}'
        print(f'Clone attempt {attempt}/3 failed: {REPO_ERROR}')
        if attempt < 3:
            time.sleep(2 ** attempt)
if REPO_READY:
    try:
        COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
        if str(REPO) not in sys.path:
            sys.path.insert(0, str(REPO))
        if INSTALL_PACKAGE or RUN_TRAINING:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
        PACKAGE_READY = True
    except Exception as exc:
        REPO_ERROR = f'{type(exc).__name__}: {exc}'
        PACKAGE_READY = False
STATUS['repo_ready'] = REPO_READY
STATUS['package_ready'] = PACKAGE_READY
print({'repo_ready': REPO_READY, 'package_ready': PACKAGE_READY, 'commit': COMMIT, 'repo_error': REPO_ERROR})

In [ ]:
ASSETS = None
TRAINING_HOME = None
VAULT_ROOT = None
VAULT_STEP = None
SIGNING_KEY = None
PACK_PARTS = ()
CURRENT_CHECKPOINT = None
CURRENT_METADATA = None
ASSET_ERRORS = []
if DRIVE_MOUNTED and REPO_READY:
    try:
        from training.colab_shared_assets import resolve_colab_training_assets
        ASSETS = resolve_colab_training_assets(DRIVE_ROOT, discovery_timeout_seconds=5, require_writable_vault=False, require_pack_parts=False)
    except Exception as exc:
        ASSET_ERRORS.append(f'resolver: {type(exc).__name__}: {exc}')
if ASSETS is not None:
    TRAINING_HOME = Path(ASSETS.training_home)
    VAULT_ROOT = Path(ASSETS.vault_root)
    VAULT_STEP = int(ASSETS.vault_step)
    SIGNING_KEY = Path(ASSETS.signing_key) if ASSETS.signing_key else None
    PACK_PARTS = tuple(Path(path) for path in ASSETS.pack_parts)
else:
    candidates = []
    if DRIVE_MOUNTED:
        for root in (DRIVE_ROOT / 'MyDrive', DRIVE_ROOT / 'My Drive'):
            for candidate in (root / TRAINING_HOME_NAME, root / 'AnRa' / TRAINING_HOME_NAME):
                if candidate.is_dir():
                    candidates.append(candidate)
        shared_root = DRIVE_ROOT / 'Shareddrives'
        if shared_root.is_dir():
            try:
                for child in shared_root.iterdir():
                    for candidate in (child / TRAINING_HOME_NAME, child / 'AnRa' / TRAINING_HOME_NAME):
                        if candidate.is_dir():
                            candidates.append(candidate)
            except OSError as exc:
                ASSET_ERRORS.append(f'shared-drive scan: {exc}')
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        checkpoint = candidate / 'anra-v4-current-full-resume.pt'
        metadata = candidate / 'anra-v4-current-full-resume.json'
        if not checkpoint.is_file() or not metadata.is_file():
            continue
        try:
            step = int(json.loads(metadata.read_text(encoding='utf-8')).get('global_step', -1))
        except Exception as exc:
            ASSET_ERRORS.append(f'{candidate}: {type(exc).__name__}: {exc}')
            continue
        if TRAINING_HOME is None or step > int(VAULT_STEP if VAULT_STEP is not None else -1):
            TRAINING_HOME = candidate
            VAULT_ROOT = candidate
            VAULT_STEP = step
            SIGNING_KEY = next((candidate / name for name in ('anra-v4-recovery-signing-keys.json', 'training-signing-keys.json') if (candidate / name).is_file()), None)
            PACK_PARTS = tuple(candidate / name for name in ('v4_phase_a_170m_seed1301.tar.gz.part00', 'v4_phase_a_170m_seed1301.tar.gz.part01') if (candidate / name).is_file())
if TRAINING_HOME is not None:
    CURRENT_CHECKPOINT = TRAINING_HOME / 'anra-v4-current-full-resume.pt'
    CURRENT_METADATA = TRAINING_HOME / 'anra-v4-current-full-resume.json'
    os.environ['ANRA_SHARED_CHECKPOINT_DIR'] = str(TRAINING_HOME)
STATUS['training_home'] = str(TRAINING_HOME) if TRAINING_HOME else None
print({'training_home': str(TRAINING_HOME) if TRAINING_HOME else None, 'vault_step': VAULT_STEP, 'pack_parts': [str(path) for path in PACK_PARTS], 'asset_errors': ASSET_ERRORS})

In [ ]:
CHECKPOINT_READY = False
CHECKPOINT_HASH_VERIFIED = False
CHECKPOINT_METADATA = {}
CHECKPOINT_ERRORS = []
SIGNING_READY = False

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if TRAINING_HOME is not None and CURRENT_CHECKPOINT is not None and CURRENT_METADATA is not None:
    try:
        CHECKPOINT_METADATA = json.loads(CURRENT_METADATA.read_text(encoding='utf-8'))
        expected_size = int(CHECKPOINT_METADATA.get('size_bytes', -1))
        actual_size = CURRENT_CHECKPOINT.stat().st_size
        size_ok = expected_size > 0 and actual_size == expected_size
        if not size_ok:
            CHECKPOINT_ERRORS.append(f'size mismatch: expected {expected_size}, found {actual_size}')
        if VERIFY_CHECKPOINT_HASH:
            actual_hash = sha256_file(CURRENT_CHECKPOINT)
            expected_hash = str(CHECKPOINT_METADATA.get('sha256', ''))
            CHECKPOINT_HASH_VERIFIED = bool(expected_hash) and actual_hash == expected_hash
            if not CHECKPOINT_HASH_VERIFIED:
                CHECKPOINT_ERRORS.append('checkpoint SHA-256 verification failed')
        CHECKPOINT_READY = size_ok
    except Exception as exc:
        CHECKPOINT_ERRORS.append(f'{type(exc).__name__}: {exc}')
    if SIGNING_KEY is None:
        SIGNING_KEY = next((TRAINING_HOME / name for name in ('anra-v4-recovery-signing-keys.json', 'training-signing-keys.json') if (TRAINING_HOME / name).is_file()), None)
    if SIGNING_KEY is not None and Path(SIGNING_KEY).is_file():
        try:
            private_keys = json.loads(Path(SIGNING_KEY).read_text(encoding='utf-8'))
            manifest_key = str(private_keys.get('manifest', ''))
            evidence_key = str(private_keys.get('evidence', ''))
            SIGNING_READY = len(manifest_key) >= 64 and len(evidence_key) >= 64
            if SIGNING_READY:
                os.environ['ANRA_MANIFEST_SIGNING_KEY'] = manifest_key
                os.environ['ANRA_EVIDENCE_SIGNING_KEY'] = evidence_key
                os.environ['ANRA_REQUIRE_SIGNED_EVIDENCE'] = '1'
        except Exception as exc:
            CHECKPOINT_ERRORS.append(f'signing identity: {type(exc).__name__}: {exc}')
STATUS['checkpoint_ready'] = CHECKPOINT_READY
STATUS['signing_ready'] = SIGNING_READY
print({'checkpoint_ready': CHECKPOINT_READY, 'hash_verified': CHECKPOINT_HASH_VERIFIED, 'signing_ready': SIGNING_READY, 'checkpoint_errors': CHECKPOINT_ERRORS})

In [ ]:
RESUME_CHECKPOINT = None
SELECTED_PACK = None
PACK_ROOT = None
LAUNCH = None
PREPARED = False
TRAINING_ERRORS = []
if not RUN_TRAINING:
    print('Preflight mode: checkpoint loading, pack extraction, and training were skipped.')
else:
    required = {'drive': DRIVE_MOUNTED, 'gpu': CUDA_READY, 'repo': REPO_READY and PACKAGE_READY, 'training_home': TRAINING_HOME is not None, 'checkpoint': CHECKPOINT_READY, 'signing': SIGNING_READY}
    missing = [name for name, ready in required.items() if not ready]
    if missing:
        TRAINING_ERRORS.append('missing prerequisites: ' + ', '.join(missing))
    else:
        try:
            from training.colab_shared_assets import resolve_colab_training_assets, stage_verified_checkpoint
            from training.colab_continuation import materialize_continuation_pack, select_continuation_pack
            from runtime.safe_load import safe_torch_load
            source = Path(CURRENT_CHECKPOINT)
            source_size = source.stat().st_size
            source_hash = sha256_file(source)
            SCRATCH.mkdir(parents=True, exist_ok=True)
            def recover_source():
                try:
                    from google.colab import drive as recovery_drive
                    recovery_drive.flush_and_unmount()
                    recovery_drive.mount(str(DRIVE_ROOT), force_remount=True)
                except Exception as exc:
                    raise RuntimeError(f'Drive remount failed: {exc}') from exc
                refreshed = resolve_colab_training_assets(DRIVE_ROOT, discovery_timeout_seconds=10, require_writable_vault=False, require_pack_parts=False)
                return Path(refreshed.training_home) / source.name
            RESUME_CHECKPOINT = stage_verified_checkpoint(source, SCRATCH / 'verified-resume.pt', expected_size=source_size, expected_sha256=source_hash, recover_source=recover_source)
            payload = safe_torch_load(RESUME_CHECKPOINT, map_location='cpu')
            if not isinstance(payload, dict):
                raise ValueError('full-resume checkpoint is not a mapping')
            counts = payload.get('continuation_token_counts', {})
            phase_a_tokens_seen = int(counts.get('A', payload.get('tokens_processed', 0))) if isinstance(counts, dict) else int(payload.get('tokens_processed', 0))
            SELECTED_PACK = select_continuation_pack(phase_a_tokens_seen, PACK_CATALOG)
            PACK_ROOT = materialize_continuation_pack(training_home=Path(TRAINING_HOME), scratch_root=SCRATCH, pack_parent=REPO / 'output' / 'v2' / 'cloud_packs', pack=SELECTED_PACK)
            LAUNCH = REPO / 'output' / 'v2' / 'launch_manifests' / f'{WORKER_ID}.json'
            artifact = SCRATCH / f'anra-v4-{WORKER_ID}.pt'
            create_command = [sys.executable, '-m', 'scripts.create_cloud_launch', '--pack-root', str(PACK_ROOT), '--output', str(LAUNCH), '--artifact-path', str(artifact), '--checkpoint-source', str(RESUME_CHECKPOINT), '--worker-id', WORKER_ID, '--runtime-estimate-hours', str(SESSION_BUDGET_MINUTES / 60), '--batch-size', str(BATCH_SIZE), '--accumulation', str(GRADIENT_ACCUMULATION)]
            subprocess.run(create_command, cwd=REPO, check=True)
            signed = json.loads(LAUNCH.read_text(encoding='utf-8'))
            if signed.get('git_commit') != COMMIT:
                raise ValueError('launch manifest commit does not match the checked-out commit')
            PREPARED = True
            print({'phase_a_tokens_seen': phase_a_tokens_seen, 'pack': SELECTED_PACK.name, 'launch': str(LAUNCH), 'run_id': signed.get('run_id')})
        except Exception as exc:
            TRAINING_ERRORS.append(f'preparation: {type(exc).__name__}: {exc}')
print({'prepared': PREPARED, 'training_errors': TRAINING_ERRORS})

In [ ]:
TRAINING_RETURN_CODE = None
if not RUN_TRAINING:
    print('PREFLIGHT COMPLETE: no training started. Set RUN_TRAINING = True only after reviewing the checks above.')
elif not PREPARED or TRAINING_ERRORS:
    print('TRAINING NOT STARTED: the preflight did not satisfy every prerequisite.')
    for error in TRAINING_ERRORS:
        print(error)
else:
    try:
        os.environ['ANRA_DURABILITY_OUTBOX'] = str(SCRATCH / 'durability-outbox')
        os.environ['ANRA_DURABILITY_REPLICAS'] = json.dumps([{'name': 'drive-vault', 'path': str(VAULT_ROOT), 'kind': 'mounted_drive_single_file', 'canonical': True}])
        os.environ['ANRA_DURABILITY_MIN_PROTECTED_REPLICAS'] = '1'
        os.environ['ANRA_DURABILITY_COPY_STREAMS'] = '2'
        os.environ['ANRA_DURABILITY_ACK_TIMEOUT_SECONDS'] = '1800'
        os.environ['ANRA_CHECKPOINT_EVERY_MIN'] = '60'
        os.environ['ANRA_DURABLE_CHECKPOINT_STEPS'] = '200'
        train_command = [sys.executable, '-u', '-m', 'training.train_unified', '--mode', 'session', '--launch-manifest', str(LAUNCH), '--prepare_data', 'never', '--post-session-eval', 'none', '--session_minutes', str(TRAINING_MINUTES), '--data_path', 'training_data/anra_training.txt']
        process = subprocess.Popen(train_command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end='', flush=True)
        TRAINING_RETURN_CODE = process.wait()
        STATUS['training_started'] = True
        if TRAINING_RETURN_CODE == 0:
            print('TRAINING SESSION COMPLETE: the protected checkpoint remains in the canonical training home.')
        else:
            print(f'TRAINER EXITED WITH STATUS {TRAINING_RETURN_CODE}; no traceback was raised by the notebook.')
    except Exception as exc:
        TRAINING_ERRORS.append(f'training: {type(exc).__name__}: {exc}')
        print(TRAINING_ERRORS[-1])

In [ ]:
STATUS.update({'training_home': str(TRAINING_HOME) if TRAINING_HOME else None, 'checkpoint_ready': CHECKPOINT_READY, 'signing_ready': SIGNING_READY, 'gpu_ready': CUDA_READY, 'repo_ready': REPO_READY, 'package_ready': PACKAGE_READY, 'drive_mounted': DRIVE_MOUNTED, 'training_started': STATUS['training_started']})
print({'status': STATUS, 'asset_errors': ASSET_ERRORS, 'checkpoint_errors': CHECKPOINT_ERRORS, 'training_errors': TRAINING_ERRORS, 'return_code': TRAINING_RETURN_CODE})